# Honda canonical generator
Transforms an original HKKM block table into TPeanuts long form. No interpolation or unit conversion is applied.

In [1]:
from pathlib import Path
import gzip, re
import pandas as pd
ROOT=Path.cwd().resolve().parents[3]
RAW=ROOT/'data/atmosphere/honda/raw/frj-0608-20-12-solmin.d.gz'
OUT=ROOT/'data/atmosphere/honda/flux/honda_flux.csv'
rows=[]; cosz=azimuth=None
pattern=re.compile(r'cosZ\s*=\s*([-+0-9.]+)\s*--\s*([-+0-9.]+).*phi_Az\s*=\s*([-+0-9.]+)\s*--\s*([-+0-9.]+)')
with gzip.open(RAW,'rt') as stream:
    for line in stream:
        match=pattern.search(line)
        if match:
            lo,hi,alo,ahi=map(float,match.groups()); cosz=(lo+hi)/2; azimuth=(alo+ahi)/2; continue
        fields=line.split()
        if cosz is None or len(fields)!=5: continue
        try: values=list(map(float,fields))
        except ValueError: continue
        for particle,flux in zip(('numu','numubar','nue','nuebar'),values[1:]):
            rows.append((values[0],cosz,azimuth,particle,flux))
table=pd.DataFrame(rows,columns=['energy_GeV','cos_zenith','azimuth_deg','particle','flux'])
OUT.parent.mkdir(parents=True,exist_ok=True); table.to_csv(OUT,index=False)
assert len(table)>0 and table.flux.ge(0).all() and table.cos_zenith.between(-1,1).all()
print(OUT, table.shape); display(table.head())

G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\data\atmosphere\honda\flux\honda_flux.csv (96960, 5)


,energy_GeV,cos_zenith,azimuth_deg,particle,flux
0,0.1000,0.95,15.0,numu,8115.6
1,0.1000,0.95,15.0,numubar,8195.4
2,0.1000,0.95,15.0,nue,3976.1
3,0.1000,0.95,15.0,nuebar,3860.1
4,0.1122,0.95,15.0,numu,7343.2
